# CS 451/651: Data-Intensive Distributed Computing (Fall 2025)
# Assignment 4: Machine Learning

## 1. Download and Unpack Data

Download and unpack the feature files referenced in the <a href="https://lintool.github.io/cs451-2025f/assignments/assignment4.html">assignment landing page</a>.
Define the `data_path` accordingly:

In [ ]:
# Change path accordingly:
data_path = "/Users/jimmylin/cs451/a4"

# Needed for later, so just import here:
import os

Sanity check, make sure you can access the files:

In [ ]:
!wc {data_path}/spam.train.txt
!wc {data_path}/spam.test.txt

## 2. Loading Training Data

Per the <a href="https://lintool.github.io/cs451-2025f/assignments/assignment4.html">assignment landing page</a>, you need to figure out how many features there are, i.e., `dim`.

<font color="red">**Below, you'll have to write some code. 
In your submission, please make sure we can clearly see the output of running the code.**</font>

In [ ]:
# codecell_2a (keep this id for tracking purposes)



Set `dim` below (obviously, set it to the correct value):

In [ ]:
# codecell_2b (keep this id for tracking purposes)

dim = -1

Your next task is to create a file called `A4_load_data.py` and in the file define a function `load_data` that has the following signature:

```python
def load_data(file: str, dim: int) -> (csr_matrix, ndarray):
```

That is, the function takes in a file (e.g., one of the feature files above) and the number of dimensions of the feature vector `dim`, and returns `X` and `y`.
As discussed in lecture, these correspond to the (desired) inputs and outputs of the classifier, respectively:

+ `X` has the type `csr_matrix` (representing the feature vectors)
+ `y` has the type `ndarray` (representing the labels: 1=spam, 0=ham)

<font color="red">**Below, you'll have to write some code, but in a separate file.**</font>

In [ ]:
from A4_load_data import load_data

If you've done the implementation above correctly, the below cell should just run.

In [ ]:
X_train, y_train = load_data(os.path.join(data_path, "spam.train.txt"), dim)

We've now loaded in the training data.
Let's sanity check.

<font color="red">**In your submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_2c (keep this id for tracking purposes)

print("=== Input ===")
print(f"Type: {type(X_train)}")
print(f"Shape: {X_train.shape}")
print("=== Output ===")
print(f"Type: {type(y_train)}")
print(f"Shape: {y_train.shape}")

## 3. Basic Classifier Training with `scikit-learn`

Let's now train a `LogisticRegression` classifier (using batch gradient descent); call this `lr_classifier`.

If you've loaded the data correctly, the following should just work.
No need to modify any code.

<font color="red">**In your submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_3a (keep this id for tracking purposes)

from sklearn.linear_model import LogisticRegression

lr_classifier = LogisticRegression(solver="saga", penalty="l2", max_iter=2000)

# Train the model
lr_classifier.fit(X_train, y_train)

# Print out basic info about the classifier
print(lr_classifier)

Let's now train a classifier using stochastic gradient descent; call this `sgd_classifier`.
As a detail, in order to get a calibrated score out of the `SGDClassifier`, we need to wrap it in another class that peforms calibration.

If you've loaded the data correctly, the following should just work.
No need to modify any code.

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_3b (keep this id for tracking purposes)

from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV

# Base classifier
sgd_base = SGDClassifier(loss="log_loss")

# Wrap with calibration
sgd_classifier = CalibratedClassifierCV(SGDClassifier(loss="log_loss"), method='sigmoid')

# Train the model
sgd_classifier.fit(X_train, y_train)

# Print out basic info about the classifier
print(sgd_classifier)

Train a third classifier, below.
Literally, just select _any_ classifier in `scikit-learn`.

Let's call it `mystery_classifier`.

<font color="red">**Below, you'll have to write some code. 
In your submission, please make sure we can clearly see the output of running the code.**</font>

In [ ]:
# codecell_3c (keep this id for tracking purposes)

# TODO: Write your code below, but do not remove any lines already in this cell.


# Print out basic info about the classifier
print(mystery_classifier)

## 4. Basic Classifier Evaluation with `scikit-learn`

Let's now load the test data:

In [ ]:
X_test, y_test = load_data(os.path.join(data_path, "spam.test.txt"), dim)

Define some helper functions:

In [ ]:
import numpy as np

def make_predictions(classifier, X):
    scores = classifier.predict_proba(X)[:, 1]
    y = (scores >= 0.5).astype(int)

    return scores, y

from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import confusion_matrix

def print_report(y_scores, y_pred, y_label):
    auc = roc_auc_score(y_label, y_scores)
    acc = accuracy_score(y_label, y_pred)

    print(f"AUC      = {auc:.6f}")
    print(f"Accuracy = {acc:.6f}")
    print("")
    print(classification_report(y_label, y_pred, digits=4))


Note that in the `make_predictions` function, we're getting both scores and the actual binary predictions from the classifier.
The scores are necessary to sweep across the threshold to determine AUC (more below).

Let's evaluate the `LogisticRegression` classifier:

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_4a (keep this id for tracking purposes)

y_scores, y_pred = make_predictions(lr_classifier, X_test)
print_report(y_scores, y_pred, y_test)

Let's evaluate the (wrapped) `SGDClassifier` classifier:

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_4b (keep this id for tracking purposes)

y_scores, y_pred = make_predictions(sgd_classifier, X_test)
print_report(y_scores, y_pred, y_test)

If you recall above, we asked you to train a third classifier.

Let's evaluate it.

You may or may not need to define a new prediction function, depending on the classifier you selected.
If you _don't_ need to write new code, you can just alias the existing `make_predictions` function.

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_4c (keep this id for tracking purposes)

# TODO: Write your code below, but do not remove any lines already in this cell.


y_scores, y_pred = make_predictions_for_mystery_classifier(mystery_classifier, X_test)
print_report(y_scores, y_pred, y_test)

AUC stands for Area Under the Curve.
The larger the value, the better classifier.
But what curve?

Let's try to understand what's going on here.

The curve we're after is a line plot where:
+ the _x_ axis shows the false positive (FP) rate.
+ the _y_ axis shows the true positive (TP) rate.

Begin by understanding what these terms mean.
The curve described above is called an ROC curve.

For each instance, the classifier emits a score between 0 and 1.
From the score, the prediction function makes a decision about the actual prediction; see the implementation of `make_predictions`.
If the score is above `>= 0.5`, the classifier predicts 1 (spam); otherwise, the classifier predicts 0 (ham).
But of course, we can vary this threshold.

Anser the below question.

<font color="red">**Answer the questions below!**</font>

// qcell_4qx1 (keep this id for tracking purposes)

**Q4.1** Imagine that we vary the threshold from small to large.
What in general is the relationship between FP and TP?

<font color="red">**Your answer here.**</font>


Now let's write a fuction that plots this ROC curve for a classifier using `matplotlib`.

The function, called `plot_roc_curve`, should take in:
+ `scores` - the scores output of the classifier
+ `y_score` - the actual labels
+ `thresholds` - an array of the thresholds, e.g., `[0.1, 0.2 ... 0.9]` (this can be an optional argument).

Implement `plot_roc_curve`.

<font color="red">**Below, you'll have to write some code!**</font>

Note that when grading we're not going to be too picky about the implementation of `plot_roc_curve`; what we're after are the actual plots.
The function is deliberately underspecified to give you some flexibility in implementation.

In [ ]:
# codecell_4d (keep this id for tracking purposes)

import matplotlib.pyplot as plt

# TODO: Write your code below, but do not remove any lines already in this cell.




Plot the ROC curve for `lr_classifier` below.

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_4e (keep this id for tracking purposes)

y_scores, y_pred = make_predictions(lr_classifier, X_test)

# If you need to, adjust thresholds.
_ = plot_roc_curve(y_scores, y_test, thresholds = [i / 1000.0 for i in range(1, 1000)])

Plot the ROC curve for `sgd_classifier` below.

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_4f (keep this id for tracking purposes)

y_scores, y_pred = make_predictions(sgd_classifier, X_test)

# If you need to, adjust thresholds.
_ = plot_roc_curve(y_scores, y_test, thresholds = [i / 100.0 for i in range(1, 100)])

Plot the ROC curve for `mystery_classifier` below.

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_4g (keep this id for tracking purposes)

y_scores, y_pred = make_predictions_for_mystery_classifier(mystery_classifier, X_test)

# If you need to, adjust thresholds.
_ = plot_roc_curve(y_scores, y_test, thresholds = [i / 1000.0 for i in range(1, 1000)])

Plot the ROC curves for all three classifiers (overlaid on the same graph, different colors) below.

<font color="red">**In your assignment submission, please make sure we can clearly see the output of the following cell.**</font>

In [ ]:
# codecell_4h (keep this id for tracking purposes)

# TODO: Write your code below, but do not remove any lines already in this cell.




<font color="red">**Answer the questions below!**</font>

// qcell_4qx2 (keep this id for tracking purposes)

**Q4.2** What do the ROC curves tell you about the differences between the output classification quality of the three classifiers?

<font color="red">**Your answer here.**</font>


## 5. Exploring SGD Non-Determinism with `scikit-learn`

Go back and run the (wrapped) `SGDClassifier` training and evaluate the predictions again.

Do you get the same results (say, AUC)?

Let's explore this a bit more. In the following cell, train 5 different `SGDClassifier` instances.
Use the "wrapped" version - just copy the code above.
At the end of the training process, they should all be in the array `classifiers`.

<font color="red">**Below, you'll have to write some code!**</font>

In [ ]:
# codecell_5a (keep this id for tracking purposes)

classifiers = []

# TODO: Write your code below, but do not remove any lines already in this cell.



i = 0
for classifier in classifiers:
    print(f"Classifier {i}: {classifier}")
    i = i + 1

Let's evaluate each of these classifiers.

The output for running the following cell should be something like this:

```
Classifier 0: AUC = 0.XXXX
Classifier 1: AUC = 0.XXXX
Classifier 2: AUC = 0.XXXX
...
```

<font color="red">**Below, you'll have to write some code!**</font>

In [ ]:
# codecell_5b (keep this id for tracking purposes)

# TODO: Write your code below, but do not remove any lines already in this cell.




Confirm that each classifier does indeed indeed have a different AUC.
That is, same dataset, different training instances lead to different AUC metrics.

The next obvious question: Is there any way to combine the classifiers?
The technical term for this is to ensemble the three different classifiers.

One very simple way is just to average the scores from each classifier instance.
That is, you get a score out of each of the classifier instances... so let's just average the scores.

Implement this below and compute the AUC of simple score averaging.

<font color="red">**Below, you'll have to write some code!**</font>

In [ ]:
# codecell_5c (keep this id for tracking purposes)

# TODO: Write your code below, but do not remove any lines already in this cell.



auc = roc_auc_score(y_test, avg_scores)
print(f"{i}-classifier ensemble using score-averaging: AUC = {auc:.4f}")

<font color="red">**Answer the questions below!**</font>


// qcell_5qx (keep this id for tracking purposes)

**Q5a** What do you observe about the AUC of the score-averaging ensemble, compared to the best individual model and the worst individual model?

<font color="red">**Your answer here.**</font>

**Q5b** Why would using this simple score-averaging ensemble be better than using any individual model?

<font color="red">**Your answer here.**</font>

**Q5c** Are there any downsides to prediction using this simple score-averaging ensemble?

<font color="red">**Your answer here.**</font>


## 6. Effect of Different Dataset Sizes

How does the AUC change given the amount training data we use?
Let's do some exploration to find out, using the `LogisticRegression` classifier.

Here's the plot we're after:
+ on the _x_ axis, plot the fraction of training data we use.
+ on the _y_ axis, plot the AUC.

Let's explore using the following fractions of the training data: `[0.1, 0.25, 0.50, 0.75, 0.90, 1.0]`

That is, we sample (say 10%) of the training data, train a classifier, and measure its AUC on the test data.
Since each trial is going to be noisy, let's run each sampling rate 3 times and average the AUC across them.
That is, 3 trials at 10% sample, 3 trial at 25% sample, etc.

Plot the graph described above.

<font color="red">**Below, you'll have to write some code. 
In your submission, please make sure we can clearly see the output of running the code.**</font>

Note that when grading we're not going to be too picky about the code; what we're after is the actual plot.

In [ ]:
# codecell_6a (keep this id for tracking purposes)

# TODO: Write your code below, but do not remove any lines already in this cell.




// qcell_6qx (keep this id for tracking purposes)

**Q6** What relationship do you observe between the amount of training data used and the classifier output quality?

<font color="red">**Your answer here.**</font>


That's it for this assignment!